# ⚽ Modelo Predictivo y de Optimización Financiera para Apuestas Deportivas
**Proyecto final · IA y Analítica Aplicada · MADN USFQ**
Equipo: Carlos Gómez · Samir Hernández · Glory Terán · Carlos Pareja

> **Objetivo principal:** poder explicar, paso a paso, cómo el modelo convierte datos de partidos en una
> probabilidad y cómo esa probabilidad se convierte en una decisión financiera (apostar o no, y cuánto).
>
> **Objetivo secundario:** la app desplegada en Hugging Face usa **exactamente este mismo código** (`src/`).

### Mapa del notebook
| # | Sección | Pregunta que responde | Responsable |
|---|---|---|---|
| 0 | Configuración | ¿Cómo lo corro en Colab? | ____ |
| 1 | Entender el negocio | ¿Qué es una cuota y qué es una "ineficiencia"? | ____ |
| 2 | Datos | ¿De dónde salen y qué contienen? | ____ |
| 3 | Variables sin fuga | ¿Qué sabe el modelo antes de cada partido? | ____ |
| 4 | Split temporal | ¿Cómo evitamos hacer trampa al evaluar? | ____ |
| 5 | Modelo XGBoost | ¿Qué tan buenas son sus probabilidades? | ____ |
| 6 | Explicabilidad (SHAP) | ¿Por qué da esa probabilidad? | ____ |
| 7 | Del modelo a la decisión | ¿Apuesto? ¿A qué? ¿Cuánto? (EV + Kelly) | ____ |
| 8 | Backtest | ¿Habríamos ganado dinero? ¿Con cuánto riesgo? | ____ |
| 9 | Conclusiones y defensa | ¿Qué decimos al profesor? | Todos |

⚠️ **Uso académico.** Las casas de apuestas tienen ventaja estructural; este proyecto mide si un modelo encuentra precios mal fijados, no garantiza ganancias.

## 0. Configuración (Colab)
Ejecuta esta celda primero. Clona el repositorio del equipo (con el código compartido en `src/`) e instala librerías.
Si trabajas en local dentro del repo, la celda lo detecta y no clona nada.

👉 **Antes de la primera vez:** reemplaza `REPO_URL` por la URL del repo de GitHub del equipo.

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/sambarsa20/apuestas-ml.git"   # repo del equipo
REPO_DIR = "apuestas-ml"

if not os.path.isdir("src"):                       # no estamos dentro del repo
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)   # traer cambios del equipo
    os.chdir(REPO_DIR)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xgboost>=2.0", "shap>=0.45"], check=False)
sys.path.insert(0, os.getcwd())
print("Carpeta de trabajo:", os.getcwd())

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, shap
from src import config, data, features, model, betting, explain

pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
plt.rcParams["figure.dpi"] = 110
CACHE_DIR = os.environ.get("DATA_DIR", "data/raw")

## 1. Entender el negocio (método Feynman)

**Una cuota es un precio de una probabilidad.** Si Bet365 paga 2.50 por la victoria local, por cada \$1 apostado
recibes \$2.50 si acierta. Para que esa apuesta sea "justa", el evento debería ocurrir 1 / 2.50 = **40%** de las veces.

**La casa siempre cobra comisión.** Si sumamos 1/cuota de los tres resultados, da más de 100% (por ejemplo 105%).
Ese 5% extra es el **margen (overround)**. Para comparar con nuestro modelo, primero lo quitamos.

**¿Qué es una ineficiencia?** Que nuestro modelo diga 48% cuando el mercado (sin margen) dice 40%.
Si nuestro modelo tiene razón, la cuota está "barata" y la apuesta tiene **valor esperado positivo**:

$$EV = p_{modelo} \times cuota - 1 = 0.48 \times 2.50 - 1 = +0.20 \;\;(\text{ganamos 20 centavos por cada \$1, en promedio})$$

**¿Cuánto apostar?** El **criterio de Kelly** dice qué fracción del capital maximiza el crecimiento de largo plazo:

$$f^* = \frac{p \times cuota - 1}{cuota - 1} = \frac{0.20}{1.50} = 13.3\%$$

El reto real del proyecto: **el mercado de apuestas es muy eficiente**. Un modelo solo aporta valor si sus
probabilidades son casi tan buenas como las del mercado, y aun así discrepa en algunos partidos.

In [ ]:
# El ejemplo anterior, con el mismo código que usa la app
odds = np.array([2.50, 3.30, 3.00])            # Local, Empate, Visitante
p_modelo = np.array([0.48, 0.27, 0.25])
print("Implícitas (con margen):", betting.implied_probs(odds).round(3), "→ suman", betting.implied_probs(odds).sum().round(3))
print("Margen de la casa      :", f"{betting.overround(odds):.1%}")
print("Prob. justas mercado   :", betting.fair_probs(odds).round(3))
print("Valor esperado         :", betting.expected_value(p_modelo, odds).round(3))
print("Kelly completo         :", betting.kelly_fraction(p_modelo, odds).round(3))

## 2. Datos
**Fuente:** [Football-Data.co.uk](https://www.football-data.co.uk) — resultados y cuotas Bet365 de las 5 grandes ligas
europeas desde 2011-12, más un archivo `fixtures.csv` con los **próximos partidos** (se actualiza martes y viernes).

> **Por qué cambiamos la fuente de la propuesta (Kaggle "European Soccer Database"):** esa base termina en 2016 y
> sus variables (ratings FIFA de jugadores) no existen para los partidos de hoy. Para cumplir el objetivo de
> **predecir partidos próximos en una app**, el modelo debe entrenarse con variables que podamos calcular hoy.
> La lógica (XGBoost → comparar con Bet365 → Kelly) es la misma de la propuesta.

In [ ]:
hist = data.load_history(cache_dir=CACHE_DIR)
hist.head()

In [ ]:
resumen = (hist.assign(Liga=hist["Div"].map(lambda d: config.LEAGUES[d].split(" (")[0]))
               .groupby("Liga")
               .agg(Partidos=("FTR", "size"),
                    Local=("FTR", lambda s: (s == "H").mean()),
                    Empate=("FTR", lambda s: (s == "D").mean()),
                    Visitante=("FTR", lambda s: (s == "A").mean())))
resumen["Margen Bet365 prom."] = (hist.dropna(subset=betting.ODDS)
                                  .assign(Liga=hist["Div"].map(lambda d: config.LEAGUES[d].split(" (")[0]))
                                  .groupby("Liga")[betting.ODDS]
                                  .apply(lambda o: betting.overround(o.to_numpy()).mean()))
resumen.style.format({"Local": "{:.1%}", "Empate": "{:.1%}", "Visitante": "{:.1%}", "Margen Bet365 prom.": "{:.2%}"})

**Cómo leerlo:** el local gana cerca de 45% de las veces (ventaja de casa) y el empate es el resultado más difícil
de predecir. El margen promedio es la "comisión" que el modelo debe superar para ganar dinero.

## 3. Variables sin fuga de información (*leakage*)

**Idea Feynman:** recorremos los partidos **en orden cronológico**, como si viviéramos cada fecha.
1. Antes del partido tomamos una **foto** del estado de los dos equipos.
2. Después del partido **actualizamos** ese estado con el resultado.

Así el modelo nunca ve el futuro. Si calculáramos, por ejemplo, el promedio de goles de la temporada completa,
estaríamos usando partidos que aún no se jugaron: el modelo parecería genial en el notebook y fallaría en la vida real.

| Variable | Qué mide (simple) |
|---|---|
| **Elo** | Un "ranking" que sube al ganar y baja al perder; ganar a un rival fuerte sube más. Cada partido mueve el Elo según qué tan sorpresivo fue el resultado. |
| `elo_diff` | Elo local + ventaja de casa (60 pts) − Elo visitante. **La variable estrella.** |
| `home_pts5`, `away_pts5` | Puntos por partido en los últimos 5 (forma reciente) |
| `*_gf5`, `*_ga5` | Goles a favor / en contra promedio últimos 5 |
| `home_pts5_home`, `away_pts5_away` | Forma específica jugando de local / de visitante |
| `div_*` | Liga (cada liga tiene distinta tasa de empates) |

Detalles del Elo: al iniciar temporada el rating vuelve 20% hacia 1500 (los planteles cambian) y un equipo
recién ascendido arranca en 1420.

In [ ]:
ds, state = features.build_dataset(hist)
print("Dataset:", ds.shape)
ds[["Date", "HomeTeam", "AwayTeam", "FTR", "elo_diff", "home_pts5", "away_pts5", "gd_diff"]].tail()

In [ ]:
# Evolución del Elo de los 4 equipos con mayor rating hoy
top = sorted(state.elo, key=state.elo.get, reverse=True)[:4]
fig, ax = plt.subplots(figsize=(10, 4))
for t in top:
    h = ds[ds["HomeTeam"] == t][["Date", "home_elo"]].rename(columns={"home_elo": "elo"})
    a = ds[ds["AwayTeam"] == t][["Date", "away_elo"]].rename(columns={"away_elo": "elo"})
    s = pd.concat([h, a]).sort_values("Date")
    ax.plot(s["Date"], s["elo"].rolling(5, min_periods=1).mean(), label=t)
ax.axhline(1500, color="gray", ls="--", lw=.8); ax.set_title("Elo antes de cada partido"); ax.legend()
plt.show()

In [ ]:
# Comprobación anti-fuga: el primer partido de cada equipo no tiene forma (NaN) porque aún no hay historia
print("Partidos sin historia de forma:", int(ds["home_pts5"].isna().sum()), "| XGBoost maneja los NaN de forma nativa.")

## 4. Split temporal

Nunca dividimos al azar: eso mezclaría partidos de 2025 en el entrenamiento y de 2015 en la prueba.
Dividimos **por temporadas**, igual que funcionaría en la vida real:

`[1ra temporada: calentamiento del Elo, se descarta] → [entrenamiento] → [validación: 1 temporada] → [prueba: 2 últimas]`

- **Validación:** decide cuántos árboles usar (*early stopping*: se para cuando deja de mejorar).
- **Prueba:** el modelo nunca la ve; ahí medimos métricas y simulamos las apuestas.

In [ ]:
train, valid, test = model.temporal_split(ds)
pd.DataFrame({
    "Conjunto": ["Entrenamiento", "Validación", "Prueba"],
    "Temporadas": [f"{d['Season'].min()} → {d['Season'].max()}" for d in (train, valid, test)],
    "Partidos": [len(d) for d in (train, valid, test)],
    "Desde": [d["Date"].min().date() for d in (train, valid, test)],
    "Hasta": [d["Date"].max().date() for d in (train, valid, test)],
})

## 5. Modelo XGBoost multiclase

**Feynman:** XGBoost construye cientos de árboles de decisión pequeños, uno tras otro. Cada árbol nuevo intenta
corregir los errores de los anteriores ("si el Elo es muy superior y el visitante viene en mala racha, sube la
probabilidad local"). Al final suma todo y lo convierte en tres probabilidades que suman 100% (función *softmax*).

Hiperparámetros conservadores (árboles poco profundos, aprendizaje lento) porque el fútbol tiene mucho ruido:
un modelo complejo memoriza casualidades.

In [ ]:
m = model.train_model(train, valid)
n_trees = model.best_n_trees(m)
print("Árboles elegidos por early stopping:", n_trees)
print("Parámetros:", {k: v for k, v in config.XGB_PARAMS.items() if k in ["max_depth", "learning_rate", "min_child_weight", "subsample"]})

test_proba = model.predict_proba(m, test)
freq = train["y"].value_counts(normalize=True).sort_index().to_numpy()
metricas = model.evaluate(test, test_proba, freq)
metricas

**Cómo leer las métricas** (en la prueba, partidos nunca vistos):
- **Log loss** y **Brier**: miden la calidad de las **probabilidades** (menor = mejor). Son las que importan para apostar,
  porque Kelly usa la probabilidad, no solo "quién gana".
- **Accuracy**: % de veces que el resultado más probable ocurrió. En fútbol, ~50% ya es bueno (el empate casi nunca es el favorito).
- **La vara es el mercado:** si nuestro log loss es peor que el de Bet365, en promedio el mercado sabe más que nosotros.
  Las "ineficiencias" que detectemos serán, en su mayoría, errores de nuestro modelo.

In [ ]:
cal = model.calibration_table(test["y"].to_numpy(), test_proba)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), sharey=True)
for ax, (cls, g) in zip(axes, cal.groupby("Clase", sort=False)):
    ax.plot([0, 1], [0, 1], "--", color="gray", lw=.8)
    ax.plot(g["Prob. predicha"], g["Frecuencia real"], "o-")
    ax.set_title(cls); ax.set_xlabel("Probabilidad predicha"); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
axes[0].set_ylabel("Frecuencia real")
fig.suptitle("Calibración: si el modelo dice 30%, ¿pasa ~30% de las veces? (sobre la diagonal = bien)")
plt.tight_layout(); plt.show()

## 6. Explicabilidad con SHAP: ¿por qué el modelo decide lo que decide?

**Feynman:** el modelo arranca con una probabilidad "promedio" para cualquier partido. Cada variable **empuja**
esa probabilidad hacia arriba o hacia abajo. SHAP (basado en teoría de juegos) reparte de forma justa cuánto
empujó cada variable. Sumando todos los empujones llegamos a la predicción de ese partido.

*(Técnicamente los empujones están en escala log-odds, antes del softmax: interpretamos signo y tamaño relativo.)*

In [ ]:
explainer = explain.make_explainer(m)
sv = explain.shap_values(explainer, test)          # (partidos, variables, clases)
imp = explain.global_importance(sv)
imp.head(10).style.format({c: "{:.3f}" for c in ["Local", "Empate", "Visitante", "Total"]})

In [ ]:
# Beeswarm para la clase "Local": cada punto es un partido; color = valor de la variable
Xt = test[features.FEATURES].rename(columns=features.FEATURE_LABELS)
shap.summary_plot(sv[:, :, 0], Xt, max_display=10, show=False, plot_size=(9, 5))
plt.title("¿Qué empuja la probabilidad de victoria LOCAL?"); plt.tight_layout(); plt.show()

**Cómo leerlo:** puntos rojos (valor alto) a la derecha = cuando esa variable es alta, sube la probabilidad local.
Si `elo_diff` alto (rojo) aparece a la derecha, el modelo aprendió algo lógico: el equipo más fuerte gana más.
Esa coherencia con el sentido común es parte de la validación del modelo.

## 7. Del modelo a la decisión (un partido real del set de prueba)

La regla de decisión (la misma de la app):
1. Quitar el margen a las cuotas → probabilidad justa del mercado.
2. Calcular el valor esperado (EV) de los 3 resultados, descartando cuotas mayores a 5 (batacazos).
3. Si el mejor EV ≥ **5%**, apostar a ese resultado; si no, **no apostar**.
4. Monto = Kelly × **1/4** (Kelly fraccional), con tope de **5%** del capital.

¿Por qué umbral y Kelly fraccional? Porque nuestra probabilidad tiene error. Kelly completo asume que la
probabilidad es exacta; si sobreestimamos un poco, apostamos de más y el capital sufre caídas enormes.

In [ ]:
odds_t = test[betting.ODDS].to_numpy(dtype=float)
ev_t = test_proba * odds_t - 1
ev_ok = np.where(odds_t <= config.MAX_ODDS, ev_t, -np.inf)
i = int(np.nanargmax(np.where(np.isfinite(ev_ok.max(1)) & (ev_ok.max(1) >= config.MIN_EV) & (ev_ok.max(1) < 0.15),
                             ev_ok.max(1), -np.inf)))     # un partido "típico" con valor
r = test.iloc[i]
d = betting.decide(test_proba[i], odds_t[i])
print(f"{r['Date']:%d/%m/%Y} · {config.LEAGUES[r['Div']]} · {r['HomeTeam']} vs {r['AwayTeam']}  (resultado real: {r['FTR']})\n")
paso = pd.DataFrame({"Cuota": odds_t[i], "1/cuota": 1 / odds_t[i], "Mercado justo": d["p_fair"],
                     "Modelo": d["p_model"], "EV": d["ev"], "Kelly completo": d["kelly_full"]},
                    index=config.CLASS_NAMES)
display(paso.style.format({"Cuota": "{:.2f}", "1/cuota": "{:.1%}", "Mercado justo": "{:.1%}",
                           "Modelo": "{:.1%}", "EV": "{:+.1%}", "Kelly completo": "{:.1%}"}))
print("Margen de la casa:", f"{d['overround']:.1%}")
print("DECISIÓN:", d["reason"])
if d["bet"] is not None:
    print(f"Monto con $1.000 de capital: ${1000 * d['stake']:.2f} ({d['stake']:.2%})")

In [ ]:
# ¿Por qué el modelo dio esa probabilidad? (explicación local)
cls = d["bet"] if d["bet"] is not None else int(np.argmax(test_proba[i]))
contrib = explain.explain_one(sv[i], r, cls)
c = contrib.iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(c["Variable"], c["Impacto"], color=np.where(c["Impacto"] > 0, "#2e7d32", "#c62828"))
ax.axvline(0, color="gray", lw=.8); ax.set_title(f"Empujones hacia '{config.CLASS_NAMES[cls]}'")
plt.tight_layout(); plt.show()
print("\n".join(explain.narrative(contrib, cls)))

## 8. Backtest: ¿habríamos ganado dinero?

Simulamos apostar en orden cronológico sobre las temporadas de prueba, con \$1.000 iniciales y cuatro estrategias:

| Estrategia | Cuánto apuesta |
|---|---|
| Apuesta fija 1% | Siempre \$10 (1% del capital inicial) |
| Kelly completo | f* del capital actual, **sin tope** (para ver su riesgo real) |
| Kelly 1/2 y 1/4 | La mitad / un cuarto de f*, con tope de 5% |

Métricas: ganancia total, **ROI** (ganancia / total apostado) y **máxima caída** (peor pérdida desde un pico: el riesgo).

In [ ]:
curves, bt = betting.backtest(test, test_proba)
fig, ax = plt.subplots(figsize=(10, 4.5))
curves.groupby(level=0).last().plot(ax=ax)
ax.axhline(1000, color="gray", ls="--", lw=.8); ax.set_ylabel("Bankroll ($)"); ax.set_title("Evolución del capital")
plt.tight_layout(); plt.show()
print(f"Apuestas con valor: {bt.attrs['n_value_bets']} | tasa de acierto: {bt.attrs['hit_rate']:.1%}")
bt

In [ ]:
# Lectura automática del resultado (para no sacar conclusiones de memoria)
mk, xg = metricas.set_index("Modelo").loc[["Mercado Bet365 (sin margen)", "XGBoost (nuestro)"], "Log loss"]
best = bt.loc[bt["Ganancia %"].idxmax()]
print(f"• Log loss modelo {xg:.4f} vs mercado {mk:.4f} → " +
      ("el modelo es MÁS preciso que el mercado." if xg < mk else
       f"el mercado es más preciso por {xg - mk:.4f}: la mayoría de 'ineficiencias' detectadas son ruido del modelo."))
print(f"• Mejor estrategia: {best['Estrategia']} con {best['Ganancia %']:+.1f}% (máx. caída {best['Máx. caída %']:.1f}%).")
kc = bt.set_index("Estrategia").loc["Kelly completo"]
print(f"• Kelly completo: {kc['Ganancia %']:+.1f}% con máxima caída de {kc['Máx. caída %']:.1f}% → muestra por qué se usa Kelly fraccional.")

## 9. Conclusiones y guía para la defensa

**El pipeline en una frase:** *datos históricos → variables calculadas solo con el pasado → XGBoost estima
probabilidades → las comparamos con el precio del mercado sin comisión → apostamos solo donde el valor esperado
supera un umbral, con un monto de Kelly fraccional.*

### Preguntas probables del profesor
| Pregunta | Respuesta corta |
|---|---|
| ¿Por qué no usaron Kaggle como decía la propuesta? | Termina en 2016 y sus variables no existen hoy; sin eso no hay predicción en vivo. La metodología no cambió. |
| ¿Cómo evitaron data leakage? | Variables calculadas cronológicamente (foto antes, actualización después) y split por temporadas. |
| ¿Por qué XGBoost y no regresión logística? | Captura relaciones no lineales e interacciones (Elo × forma), maneja NaN y es explicable con SHAP. Se puede comparar con logística como baseline. |
| ¿Por qué quitar el margen? | Sin quitarlo, todas las probabilidades del mercado están infladas y veríamos valor donde no hay. |
| ¿Por qué Kelly fraccional? | Kelly completo asume probabilidades exactas; con error de estimación produce caídas enormes (lo muestra el backtest). |
| ¿Funciona para ganar dinero? | Responder con los números del backtest, no con optimismo. Que el mercado sea difícil de batir **también es un hallazgo**. |

### Posibles mejoras (trabajo futuro)
- Agregar cuotas de cierre y de otras casas (Pinnacle) para medir eficiencia del mercado.
- Calibrar probabilidades (isotónica) y comparar con regresión logística.
- Variables de goles esperados (xG), lesiones y calendario.

### Conexión con la app
La app de Hugging Face importa los mismos módulos `src/` y repite estos pasos con los datos del día;
la pestaña **Partidos próximos** aplica la sección 7 a cada partido de `fixtures.csv`.